In [4]:
import json
import os

# --- KONFIGURATION ---
# Passe diese Pfade an deine lokale Ordnerstruktur an
ORIGINAL_FILE = '../data/annotations.json'
PROTOTYPE_FILE = '../data/prototype_annotations.json'

# Deine lokal vorhandenen IDs (G000_IMG000 bis G000_IMG102)
MY_IDS = set(range(103))

print("Starte Extraktion... Lade Originaldatei (das kann einen Moment dauern).")

# 1. ORIGINAL LADEN
try:
    with open(ORIGINAL_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)
except FileNotFoundError:
    print(f"Fehler: Die Datei {ORIGINAL_FILE} wurde nicht gefunden.")
    exit()

# 2. ZIEL-STRUKTUR INITIALISIEREN (Plan Schritt 2)
# Wir kopieren 'info' und 'categories' direkt, um IDs konsistent zu halten.
new_data = {
    "info": data.get("info", {}),
    "categories": data.get("categories", []),
    "images": [],
    "annotations": {"pieces": []},
    "splits": {
        # Flache Struktur für Code-Teile wie annotations['splits']['train']
        "train": {"image_ids": []},
        "val": {"image_ids": []},
        "test": {"image_ids": []},
        # Verschachtelte Struktur für chessred2k Pfade
        "chessred2k": {
            "train": {"image_ids": []},
            "val": {"image_ids": []},
            "test": {"image_ids": []}
        }
    }
}

# 3. DATEN EXTRAHIEREN (Plan Schritt 3)
# Bilder filtern
for img in data.get("images", []):
    if img["id"] in MY_IDS:
        new_data["images"].append(img)

# Pieces (Anmerkungen) filtern
for anno in data.get("annotations", {}).get("pieces", []):
    if anno["image_id"] in MY_IDS:
        new_data["annotations"]["pieces"].append(anno)

# 4. SPLITS DYNAMISCH ZUWEISEN (Plan Schritt 4)
# Wir sortieren die IDs, um eine saubere Aufteilung zu gewährleisten
sorted_ids = sorted(list(MY_IDS))
for img_id in sorted_ids:
    # Aufteilung: 0-79 (Train), 80-92 (Val), 93-102 (Test)
    if img_id < 80:
        target = "train"
    elif img_id < 93:
        target = "val"
    else:
        target = "test"
    
    # Doppelte Zuweisung für maximale Kompatibilität mit deinem Notebook
    new_data["splits"][target]["image_ids"].append(img_id)
    new_data["splits"]["chessred2k"][target]["image_ids"].append(img_id)

# 5. SPEICHERN (Plan Schritt 5)
with open(PROTOTYPE_FILE, 'w', encoding='utf-8') as f:
    json.dump(new_data, f, indent=2)

print("-" * 30)
print(f"ERFOLG: {PROTOTYPE_FILE} wurde erstellt.")
print(f"Bilder extrahiert: {len(new_data['images'])}")
print(f"Piece-Annotations extrahiert: {len(new_data['annotations']['pieces'])}")
print(f"Train/Val/Test Aufteilung: {len(new_data['splits']['train']['image_ids'])} / "
      f"{len(new_data['splits']['val']['image_ids'])} / "
      f"{len(new_data['splits']['test']['image_ids'])}")
print("-" * 30)

Starte Extraktion... Lade Originaldatei (das kann einen Moment dauern).
------------------------------
ERFOLG: ../data/prototype_annotations.json wurde erstellt.
Bilder extrahiert: 103
Piece-Annotations extrahiert: 2099
Train/Val/Test Aufteilung: 80 / 13 / 10
------------------------------
